# Log analysis

# Coding problem, 4.4.b: Brute-force log correlator

**Skill:** Detect Attacks · **Maps to:** 4.4.B · **Language:** Python 3 (pytest)

## Problem
An authentication log records every login attempt as an event with a username, a timestamp (in
seconds), and a result (`"SUCCESS"` or `"FAILED"`). A brute-force attack shows up as a cluster of
failed attempts for the same user, tightly packed in time. Build the correlator that scans the
log and flags every username showing this pattern.

## Function contract
Implement `def solve(entries: list) -> list` in the code editor (see the Reference section below).

## Input
A list of dicts, each with keys `ts` (int, seconds since some reference point), `user` (str),
`result` (str, `"SUCCESS"` or `"FAILED"`). Entries are not guaranteed to already be sorted by
timestamp.

## Output
A sorted list of usernames (str) flagged for a brute-force pattern.

## Detection rule
A user is flagged if there exist **5 or more `"FAILED"` entries for that user** where the time
span from the earliest to the latest of those 5 (or more) entries is **at most 60 seconds**
(inclusive). In other words: sort that user's failed-attempt timestamps ascending; if any 5
consecutive failed timestamps (in that sorted order) span 60 seconds or less
(`timestamps[i+4] - timestamps[i] <= 60`), the user is flagged. `SUCCESS` entries do not count
toward the failure count and do not reset or break a streak of failures, only `FAILED` entries
matter for this rule.

## Examples
```
[{"ts": 0, "user": "jsmith", "result": "FAILED"},
 {"ts": 13, "user": "jsmith", "result": "FAILED"},
 {"ts": 21, "user": "jsmith", "result": "FAILED"},
 {"ts": 30, "user": "jsmith", "result": "FAILED"},
 {"ts": 38, "user": "jsmith", "result": "FAILED"},
 {"ts": 46, "user": "jsmith", "result": "SUCCESS"},
 {"ts": 100, "user": "agarcia", "result": "FAILED"},
 {"ts": 1662, "user": "agarcia", "result": "FAILED"}]
    -> ["jsmith"]
```
*(jsmith has 5 failures spanning 0-38s = 38s <= 60s -> flagged. agarcia has only 2 failures,
1562s apart -> not flagged.)*

## Constraints
- Deterministic, offline, standard library only.
- Input entries may arrive out of timestamp order; sort within your implementation.
- Return a de-duplicated, alphabetically sorted list of usernames.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(entries: list) -> list
```

### Input (`entries`)
A list of dicts, each an authentication log event:

| key | type | meaning |
|---|---|---|
| `ts` | int | timestamp in seconds (not guaranteed sorted across the list) |
| `user` | str | username the attempt was made against |
| `result` | str | `"SUCCESS"` or `"FAILED"` |

### Output (`result`)
A de-duplicated, alphabetically sorted list of `str` usernames flagged for a brute-force pattern.

### Detection rule
For each user, collect all `FAILED` timestamps and sort them ascending. The user is flagged if
any 5 consecutive timestamps in that sorted list (`t[i]` through `t[i+4]`) satisfy
`t[i+4] - t[i] <= 60`. A user needs at least 5 total failed attempts to be eligible; `SUCCESS`
entries are ignored entirely for this rule (they do not reset or interrupt a failure streak).

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only unless the hidden grading tests: allow_deps` lists otherwise.
- Input order is not guaranteed; sort per-user failed timestamps before evaluating the rule.
- The 60-second window is inclusive (`<= 60`, not `< 60`).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 4.4.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""


def solve(entries: list) -> list:
    """Flag usernames showing a brute-force pattern: 5+ FAILED attempts within any
    60-second window.

    Args:
        entries: list of dicts with keys ts (int), user (str), result ("SUCCESS"/"FAILED").
            Not guaranteed sorted by timestamp.

    Returns:
        Alphabetically sorted list of unique usernames flagged for brute-force.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_example_1_prompt_scenario():
    entries = [
        {"ts": 0, "user": "jsmith", "result": "FAILED"},
        {"ts": 13, "user": "jsmith", "result": "FAILED"},
        {"ts": 21, "user": "jsmith", "result": "FAILED"},
        {"ts": 30, "user": "jsmith", "result": "FAILED"},
        {"ts": 38, "user": "jsmith", "result": "FAILED"},
        {"ts": 46, "user": "jsmith", "result": "SUCCESS"},
        {"ts": 100, "user": "agarcia", "result": "FAILED"},
        {"ts": 1662, "user": "agarcia", "result": "FAILED"},
    ]
    assert impl.solve(entries) == ["jsmith"]


def test_example_2_no_flags():
    entries = [
        {"ts": 0, "user": "mchen", "result": "SUCCESS"},
        {"ts": 500, "user": "rpatel", "result": "FAILED"},
        {"ts": 1700, "user": "rpatel", "result": "FAILED"},
    ]
    assert impl.solve(entries) == []


# ---- hidden / edge cases -------------------------------------------------
def test_edge_empty_entries():
    assert impl.solve([]) == []


def test_fewer_than_five_failures_never_flagged():
    entries = [
        {"ts": t, "user": "twilson", "result": "FAILED"} for t in [0, 5, 10, 15]
    ]
    assert impl.solve(entries) == []


def test_five_failures_exactly_at_60s_boundary_flags():
    entries = [
        {"ts": t, "user": "bwright", "result": "FAILED"} for t in [0, 15, 30, 45, 60]
    ]
    assert impl.solve(entries) == ["bwright"]


def test_five_failures_just_over_60s_does_not_flag():
    entries = [
        {"ts": t, "user": "bwright", "result": "FAILED"} for t in [0, 15, 30, 45, 61]
    ]
    assert impl.solve(entries) == []


def test_out_of_order_input_still_correlated_correctly():
    entries = [
        {"ts": 46, "user": "jsmith", "result": "SUCCESS"},
        {"ts": 21, "user": "jsmith", "result": "FAILED"},
        {"ts": 0, "user": "jsmith", "result": "FAILED"},
        {"ts": 38, "user": "jsmith", "result": "FAILED"},
        {"ts": 13, "user": "jsmith", "result": "FAILED"},
        {"ts": 30, "user": "jsmith", "result": "FAILED"},
    ]
    assert impl.solve(entries) == ["jsmith"]


def test_sliding_window_finds_cluster_not_just_first_and_last():
    # 7 failures total; the FIRST-to-LAST span is 200s (too wide), but a 5-in-a-row
    # cluster from index 1..5 spans only 40s and should still trigger the flag.
    entries = [
        {"ts": 0, "user": "cluster", "result": "FAILED"},
        {"ts": 100, "user": "cluster", "result": "FAILED"},
        {"ts": 110, "user": "cluster", "result": "FAILED"},
        {"ts": 120, "user": "cluster", "result": "FAILED"},
        {"ts": 130, "user": "cluster", "result": "FAILED"},
        {"ts": 140, "user": "cluster", "result": "FAILED"},
        {"ts": 200, "user": "cluster", "result": "FAILED"},
    ]
    assert impl.solve(entries) == ["cluster"]


def test_multiple_users_result_sorted_alphabetically():
    entries = []
    for t in [0, 10, 20, 30, 40]:
        entries.append({"ts": t, "user": "zach", "result": "FAILED"})
    for t in [0, 10, 20, 30, 40]:
        entries.append({"ts": t, "user": "amy", "result": "FAILED"})
    assert impl.solve(entries) == ["amy", "zach"]


def test_success_entries_do_not_count_as_failures():
    entries = [
        {"ts": 0, "user": "user1", "result": "SUCCESS"},
        {"ts": 5, "user": "user1", "result": "FAILED"},
        {"ts": 10, "user": "user1", "result": "SUCCESS"},
        {"ts": 15, "user": "user1", "result": "FAILED"},
        {"ts": 20, "user": "user1", "result": "SUCCESS"},
        {"ts": 25, "user": "user1", "result": "FAILED"},
    ]
    assert impl.solve(entries) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
